In [ ]:

# ============================================================
# DATA QUALITY NOTEBOOK - REJECTED LOANS
# Dataset: sample_rejected_2007_to_2018Q4.csv
# Objective: Diagnose, visualize, clean, document and validate
# IMPORTANT: Original/raw files are NEVER modified.
# ============================================================

# ============================================================
# CELL 1 - IMPORT LIBRARIES & CONFIGURATION
# ============================================================

import os
import re
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

PROJECT_DIR = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject"
INPUT_DIR = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject\data\Sampled"
CLEANED_DIR = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject\data\Cleaned"
REPORT_ROOT = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject\docs\Data Cleaning"

INPUT_FILES = {
    "50K_Sample": os.path.join(INPUT_DIR, "sample_rejected_2007_to_2018Q4.csv"),
}

# All diagnostic reports/charts are stored under docs\\Data Cleaning.
OUTPUT_ROOT = os.path.join(REPORT_ROOT, "Rejected")
os.makedirs(CLEANED_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(REPORT_ROOT, exist_ok=True)

# Small yearly groups will be marked as low-volume.
MIN_ROWS_PER_YEAR = 30

sns.set_theme(style="whitegrid")


# ============================================================
# CELL 2 - HELPER FUNCTIONS
# ============================================================

def save_plot(filename):
    """Save the current plot."""
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


def add_log(log_list, stage, column, issue, action, rows=0, details=""):
    log_list.append({
        "Stage": stage,
        "Column": column,
        "Issue": issue,
        "Action": action,
        "Affected_Rows": int(rows),
        "Details": details
    })


def parse_dti(series):
    """Convert DTI percentage strings to numeric values."""
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def clean_text(series):
    """Remove leading/trailing and repeated whitespace."""
    return (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )


def detect_iqr_outliers(series):
    """Return IQR outlier flags without changing the values."""
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()

    if len(valid) < 4:
        return pd.Series(False, index=series.index)

    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        return pd.Series(False, index=series.index)

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return (numeric < lower) | (numeric > upper)


def detect_category_variants(series):
    """
    Detect different raw labels that become identical
    after lowercase + whitespace normalization.
    """
    values = series.dropna().astype(str)

    if values.empty:
        return pd.DataFrame(
            columns=["Normalized_Value", "Original_Variants", "Variant_Count"]
        )

    temp = pd.DataFrame({"Original": values})
    temp["Normalized"] = (
        temp["Original"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    grouped = (
        temp.groupby("Normalized")["Original"]
        .agg(lambda x: sorted(set(x)))
        .reset_index()
    )

    grouped["Variant_Count"] = grouped["Original"].apply(len)

    result = grouped[grouped["Variant_Count"] > 1].copy()

    result = result.rename(columns={
        "Normalized": "Normalized_Value",
        "Original": "Original_Variants"
    })

    return result


# ============================================================
# CELL 3 - PROCESS EACH DATASET
# ============================================================

def run_data_quality_pipeline(dataset_name, input_path):

    print("\n" + "=" * 70)
    print("PROCESSING:", dataset_name)
    print("=" * 70)

    if not os.path.exists(input_path):
        print("File not found:", input_path)
        return

    output_dir = os.path.join(OUTPUT_ROOT, dataset_name)
    charts_dir = os.path.join(output_dir, "Before_Cleaning")
    after_dir = os.path.join(output_dir, "After_Cleaning")

    os.makedirs(charts_dir, exist_ok=True)
    os.makedirs(after_dir, exist_ok=True)

    # Read the raw file. Never overwrite it.
    raw = pd.read_csv(input_path, low_memory=False)

    # Preserve an independent copy for comparison.
    df = raw.copy(deep=True)

    print("Rows:", len(raw))
    print("Columns:", len(raw.columns))

    # Keep a log for every diagnostic and transformation.
    cleaning_log = []

    # Normalize column names in the working copy only.
    original_columns = list(df.columns)

    df.columns = (
        df.columns.astype(str)
        .str.strip()
    )

    if list(df.columns) != original_columns:
        add_log(
            cleaning_log,
            "Preparation",
            "Column Names",
            "Whitespace in column names",
            "Trimmed column names in working copy",
            len(df),
            "Raw file was not changed."
        )

    # --------------------------------------------------------
    # CELL 4 - DIAGNOSTIC DATA PREPARATION
    # This section creates diagnostic copies only.
    # No cleaning is performed here.
    # --------------------------------------------------------

    required_columns = [
        "Amount Requested",
        "Application Date",
        "Loan Title",
        "Risk_Score",
        "Debt-To-Income Ratio",
        "Zip Code",
        "State",
        "Employment Length",
        "Policy Code"
    ]

    missing_columns = [
        c for c in required_columns if c not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing expected columns: {missing_columns}"
        )

    # Parse diagnostic dates without changing the original column.
    diagnostic_date = pd.to_datetime(
        df["Application Date"],
        errors="coerce"
    )

    # Parse DTI in a separate diagnostic series.
    diagnostic_dti = parse_dti(df["Debt-To-Income Ratio"])

    diagnostic_numeric = {
        "Amount Requested": pd.to_numeric(
            df["Amount Requested"], errors="coerce"
        ),
        "Risk_Score": pd.to_numeric(
            df["Risk_Score"], errors="coerce"
        ),
        "Debt-To-Income Ratio": diagnostic_dti,
        "Policy Code": pd.to_numeric(
            df["Policy Code"], errors="coerce"
        )
    }

    # --------------------------------------------------------
    # CELL 5 - NULL ANALYSIS
    # --------------------------------------------------------

    null_report = pd.DataFrame({
        "Column": df.columns,
        "Missing_Count": df.isna().sum().values,
        "Missing_Percentage": (
            df.isna().mean().values * 100
        ).round(2)
    }).sort_values(
        "Missing_Percentage",
        ascending=False
    )

    null_report.to_csv(
        os.path.join(output_dir, "Null_Analysis.csv"),
        index=False
    )

    # Graph 1: Missing values by column.
    plt.figure(figsize=(12, 6))

    sns.barplot(
        data=null_report,
        x="Missing_Percentage",
        y="Column"
    )

    plt.title("Missing Values Percentage - Before Cleaning")
    plt.xlabel("Missing Values (%)")
    plt.ylabel("Column")

    save_plot(
        os.path.join(charts_dir, "01_Missing_Values.png")
    )

    # Graph 2: Co-missing columns.
    # A value of 1 means both columns are missing in that row.
    missing_matrix = df.isna().astype(int)

    if len(df.columns) > 1:

        co_missing = missing_matrix.T.dot(missing_matrix)

        plt.figure(figsize=(10, 8))

        sns.heatmap(
            co_missing,
            annot=True,
            fmt="g",
            cmap="Blues"
        )

        plt.title("Co-Missing Values Count")
        plt.xlabel("Columns")
        plt.ylabel("Columns")

        save_plot(
            os.path.join(charts_dir, "02_Co_Missing_Heatmap.png")
        )

        co_missing.to_csv(
            os.path.join(output_dir, "Co_Missing_Matrix.csv")
        )

    # --------------------------------------------------------
    # CELL 6 - OUTLIER ANALYSIS
    # --------------------------------------------------------

    outlier_flags = pd.DataFrame(index=df.index)

    for col, values in diagnostic_numeric.items():

        outlier_flags[col + "_IQR_Outlier"] = (
            detect_iqr_outliers(values)
        )

    # Additional domain-review flags.
    outlier_flags["Amount_Non_Positive"] = (
        diagnostic_numeric["Amount Requested"] <= 0
    )

    outlier_flags["Risk_Score_Zero"] = (
        diagnostic_numeric["Risk_Score"] == 0
    )

    outlier_flags["Risk_Score_Outside_300_850"] = (
        (diagnostic_numeric["Risk_Score"] < 300) |
        (diagnostic_numeric["Risk_Score"] > 850)
    )

    outlier_flags["DTI_Negative"] = diagnostic_dti < 0
    outlier_flags["DTI_Above_100"] = diagnostic_dti > 100

    outlier_flags["DTI_Extreme_Above_1000"] = (
        diagnostic_dti > 1000
    )

    outlier_flags["Invalid_Application_Date"] = (
        diagnostic_date.isna()
    )

    outlier_flags.to_csv(
        os.path.join(output_dir, "Outlier_And_Issue_Flags.csv"),
        index=False
    )

    outlier_summary = pd.DataFrame({
        "Issue": outlier_flags.columns,
        "Affected_Rows": outlier_flags.sum().values,
        "Percentage": (
            outlier_flags.mean().values * 100
        ).round(2)
    }).sort_values(
        "Affected_Rows",
        ascending=False
    )

    outlier_summary.to_csv(
        os.path.join(output_dir, "Outlier_Summary.csv"),
        index=False
    )

    # Graph 3: Numeric distributions / boxplots.
    numeric_names = list(diagnostic_numeric.keys())

    fig, axes = plt.subplots(
        len(numeric_names),
        1,
        figsize=(12, 4 * len(numeric_names))
    )

    for ax, col in zip(axes, numeric_names):

        sns.boxplot(
            x=diagnostic_numeric[col],
            ax=ax
        )

        ax.set_title(f"Boxplot - {col}")

    save_plot(
        os.path.join(charts_dir, "03_Numeric_Outliers.png")
    )

    # Graph 4: Issue counts.
    plt.figure(figsize=(12, 6))

    sns.barplot(
        data=outlier_summary,
        x="Affected_Rows",
        y="Issue"
    )

    plt.title("Detected Issues Before Cleaning")
    plt.xlabel("Affected Rows")

    save_plot(
        os.path.join(charts_dir, "04_Issue_Counts.png")
    )

    # --------------------------------------------------------
    # CELL 7 - INCONSISTENT CATEGORY ANALYSIS
    # --------------------------------------------------------

    category_columns = [
        "Loan Title",
        "Zip Code",
        "State",
        "Employment Length"
    ]

    category_reports = []

    for col in category_columns:

        variants = detect_category_variants(df[col])

        if not variants.empty:

            variants.insert(0, "Column", col)

            category_reports.append(variants)

    if category_reports:

        category_report = pd.concat(
            category_reports,
            ignore_index=True
        )

    else:

        category_report = pd.DataFrame(
            columns=[
                "Column",
                "Normalized_Value",
                "Original_Variants",
                "Variant_Count"
            ]
        )

    category_report.to_csv(
        os.path.join(
            output_dir,
            "Category_Inconsistencies.csv"
        ),
        index=False
    )

    # Graph 5: Category frequency.
    fig, axes = plt.subplots(
        len(category_columns),
        1,
        figsize=(13, 5 * len(category_columns))
    )

    for ax, col in zip(axes, category_columns):

        counts = (
            df[col]
            .astype("string")
            .value_counts(dropna=False)
            .head(15)
        )

        counts.plot(
            kind="bar",
            ax=ax
        )

        ax.set_title(f"Top Categories - {col}")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)

    save_plot(
        os.path.join(charts_dir, "05_Category_Distributions.png")
    )

    # --------------------------------------------------------
    # CELL 8 - ISSUE OVERLAP ANALYSIS
    # --------------------------------------------------------

    issue_overlap = outlier_flags.astype(int).copy()

    issue_overlap["Missing_Loan_Title"] = (
        df["Loan Title"].isna().astype(int)
    )

    issue_overlap["Missing_Risk_Score"] = (
        df["Risk_Score"].isna().astype(int)
    )

    issue_overlap["Missing_Employment_Length"] = (
        df["Employment Length"].isna().astype(int)
    )

    issue_overlap["Missing_Policy_Code"] = (
        df["Policy Code"].isna().astype(int)
    )

    issue_overlap["Missing_Application_Date"] = (
        diagnostic_date.isna().astype(int)
    )

    issue_overlap["Missing_DTI"] = (
        diagnostic_dti.isna().astype(int)
    )

    issue_overlap["Issues_Per_Row"] = (
        issue_overlap.sum(axis=1)
    )

    issue_overlap.to_csv(
        os.path.join(output_dir, "Issue_Overlap_By_Row.csv"),
        index=False
    )

    overlap_corr = issue_overlap.drop(
        columns=["Issues_Per_Row"]
    ).corr()

    plt.figure(figsize=(14, 10))

    sns.heatmap(
        overlap_corr,
        cmap="coolwarm",
        center=0,
        annot=True,
        fmt=".2f"
    )

    plt.title("Issue Overlap Correlation")

    save_plot(
        os.path.join(charts_dir, "06_Issue_Overlap.png")
    )

    # Graph 7: Number of issues per row.
    plt.figure(figsize=(10, 5))

    issue_overlap["Issues_Per_Row"].value_counts().sort_index().plot(
        kind="bar"
    )

    plt.title("Number of Issues Per Row")
    plt.xlabel("Issues per Row")
    plt.ylabel("Number of Records")

    save_plot(
        os.path.join(charts_dir, "07_Issues_Per_Row.png")
    )

    # --------------------------------------------------------
    # CELL 9 - REDUNDANT COLUMN PAIRS
    # --------------------------------------------------------

    redundant_pairs = []

    # Exact duplicate columns.
    for col1, col2 in combinations(df.columns, 2):

        equal_mask = (
            df[col1].eq(df[col2]) |
            (df[col1].isna() & df[col2].isna())
        )

        if equal_mask.all():

            redundant_pairs.append({
                "Column_1": col1,
                "Column_2": col2,
                "Relationship": "Exact duplicate columns",
                "Similarity": 1.0
            })

    # Numeric highly correlated pairs.
    numeric_df = pd.DataFrame(diagnostic_numeric)

    corr = numeric_df.corr()

    for col1, col2 in combinations(corr.columns, 2):

        value = corr.loc[col1, col2]

        if pd.notna(value) and abs(value) >= 0.995:

            redundant_pairs.append({
                "Column_1": col1,
                "Column_2": col2,
                "Relationship": "Very high numeric correlation",
                "Similarity": round(float(value), 5)
            })

    redundant_report = pd.DataFrame(redundant_pairs)

    redundant_report.to_csv(
        os.path.join(output_dir, "Redundant_Column_Pairs.csv"),
        index=False
    )

    # Constant columns.
    constant_columns = [
        col for col in df.columns
        if df[col].nunique(dropna=False) <= 1
    ]

    pd.DataFrame({
        "Constant_Column": constant_columns
    }).to_csv(
        os.path.join(output_dir, "Constant_Columns.csv"),
        index=False
    )

    # --------------------------------------------------------
    # CELL 10 - TIMELINE ANALYSIS
    # --------------------------------------------------------

    timeline = pd.DataFrame({
        "Application_Date": diagnostic_date,
        "Missing_Risk_Score": df["Risk_Score"].isna(),
        "Missing_Loan_Title": df["Loan Title"].isna(),
        "Missing_Employment_Length": df["Employment Length"].isna(),
        "Missing_Policy_Code": df["Policy Code"].isna(),
        "Missing_DTI": diagnostic_dti.isna(),
        "Invalid_Date": diagnostic_date.isna(),
        "DTI_Negative": diagnostic_dti < 0,
        "DTI_Above_100": diagnostic_dti > 100,
        "Risk_Score_Zero": diagnostic_numeric["Risk_Score"] == 0
    })

    timeline["Year"] = timeline["Application_Date"].dt.year
    timeline["Month"] = timeline["Application_Date"].dt.to_period("M")

    valid_timeline = timeline.dropna(subset=["Application_Date"])

    # Monthly missingness and issue rates.
    monthly = (
        valid_timeline
        .groupby("Month")
        .agg(
            Records=("Application_Date", "size"),
            Missing_Risk_Score=("Missing_Risk_Score", "mean"),
            Missing_Loan_Title=("Missing_Loan_Title", "mean"),
            Missing_Employment_Length=("Missing_Employment_Length", "mean"),
            Missing_Policy_Code=("Missing_Policy_Code", "mean"),
            Missing_DTI=("Missing_DTI", "mean"),
            DTI_Negative=("DTI_Negative", "mean"),
            DTI_Above_100=("DTI_Above_100", "mean"),
            Risk_Score_Zero=("Risk_Score_Zero", "mean")
        )
        .reset_index()
    )

    rate_columns = [
        c for c in monthly.columns
        if c not in ["Month", "Records"]
    ]

    monthly[rate_columns] = monthly[rate_columns] * 100

    monthly["Low_Volume_Flag"] = (
        monthly["Records"] < MIN_ROWS_PER_YEAR
    )

    monthly.to_csv(
        os.path.join(output_dir, "Monthly_Issue_Timeline.csv"),
        index=False
    )

    # Yearly timeline.
    yearly = (
        valid_timeline
        .groupby("Year")
        .agg(
            Records=("Application_Date", "size"),
            Missing_Risk_Score=("Missing_Risk_Score", "mean"),
            Missing_Loan_Title=("Missing_Loan_Title", "mean"),
            Missing_Employment_Length=("Missing_Employment_Length", "mean"),
            Missing_Policy_Code=("Missing_Policy_Code", "mean"),
            Missing_DTI=("Missing_DTI", "mean"),
            DTI_Negative=("DTI_Negative", "mean"),
            DTI_Above_100=("DTI_Above_100", "mean"),
            Risk_Score_Zero=("Risk_Score_Zero", "mean")
        )
        .reset_index()
    )

    yearly[rate_columns] = yearly[rate_columns] * 100

    yearly["Low_Volume_Flag"] = (
        yearly["Records"] < MIN_ROWS_PER_YEAR
    )

    yearly.to_csv(
        os.path.join(output_dir, "Yearly_Issue_Timeline.csv"),
        index=False
    )

    # Graph 8: Missingness over time.
    plt.figure(figsize=(14, 6))

    for col in [
        "Missing_Risk_Score",
        "Missing_Loan_Title",
        "Missing_Employment_Length",
        "Missing_Policy_Code",
        "Missing_DTI"
    ]:

        plt.plot(
            monthly["Month"].astype(str),
            monthly[col],
            marker="o",
            label=col
        )

    plt.title("Monthly Missingness Timeline")
    plt.xlabel("Application Month")
    plt.ylabel("Missing Values (%)")
    plt.xticks(rotation=90)
    plt.legend()

    save_plot(
        os.path.join(charts_dir, "08_Missingness_Timeline.png")
    )

    # Graph 9: Other issues over time.
    plt.figure(figsize=(14, 6))

    for col in [
        "DTI_Negative",
        "DTI_Above_100",
        "Risk_Score_Zero"
    ]:

        plt.plot(
            monthly["Month"].astype(str),
            monthly[col],
            marker="o",
            label=col
        )

    plt.title("Monthly Issue Timeline")
    plt.xlabel("Application Month")
    plt.ylabel("Issue Rate (%)")
    plt.xticks(rotation=90)
    plt.legend()

    save_plot(
        os.path.join(charts_dir, "09_Other_Issues_Timeline.png")
    )

    # --------------------------------------------------------
    # CELL 11 - DOCUMENT DIAGNOSTIC FINDINGS
    # --------------------------------------------------------

    add_log(
        cleaning_log,
        "Diagnosis",
        "All Columns",
        "Missing values",
        "Reported and preserved as null",
        int(df.isna().sum().sum()),
        "No missing values were filled with zero."
    )

    add_log(
        cleaning_log,
        "Diagnosis",
        "All Rows",
        "Exact duplicate rows",
        "Reported only; no rows deleted",
        int(df.duplicated().sum()),
        "Deletion requires a documented decision."
    )

    add_log(
        cleaning_log,
        "Diagnosis",
        "Numeric Columns",
        "IQR outliers",
        "Flagged for review; original values preserved",
        int(outlier_flags.filter(like="_IQR_Outlier").sum().sum()),
        "IQR is a statistical screening rule, not proof of invalid data."
    )

    add_log(
        cleaning_log,
        "Diagnosis",
        "Categorical Columns",
        "Inconsistent category variants",
        "Reported for review",
        len(category_report),
        "No categories merged automatically."
    )

    add_log(
        cleaning_log,
        "Diagnosis",
        "Application Date",
        "Timeline coverage",
        "Monthly and yearly issue rates calculated",
        len(valid_timeline),
        "Low-volume periods are flagged."
    )

    # --------------------------------------------------------
    # CELL 12 - CLEANING STARTS HERE
    # All pre-cleaning graphs and diagnostics have been saved.
    # --------------------------------------------------------

    cleaned = df.copy(deep=True)

    # A. Trim text fields, but preserve capitalization and meaning.
    text_cols = cleaned.select_dtypes(
        include=["object", "string"]
    ).columns

    for col in text_cols:

        before = cleaned[col].copy()

        after = clean_text(cleaned[col])

        changed = (
            before.fillna("<NA>").astype(str) !=
            after.fillna("<NA>").astype(str)
        )

        changed_count = int(changed.sum())

        cleaned[col] = after

        if changed_count > 0:

            add_log(
                cleaning_log,
                "Cleaning",
                col,
                "Leading/trailing or repeated whitespace",
                "Trimmed and normalized whitespace",
                changed_count,
                "Text meaning and capitalization preserved."
            )

    # B. Convert application date to datetime.
    before_date = cleaned["Application Date"].copy()

    cleaned["Application Date"] = pd.to_datetime(
        cleaned["Application Date"],
        errors="coerce"
    )

    date_conversion_failures = int(
        before_date.notna().sum() -
        cleaned["Application Date"].notna().sum()
    )

    add_log(
        cleaning_log,
        "Cleaning",
        "Application Date",
        "Date type",
        "Converted to datetime; unparseable values remain null",
        date_conversion_failures,
        "Original date values were not used to overwrite the raw file."
    )

    # C. Convert DTI to numeric after removing percentage symbol.
    before_dti = cleaned["Debt-To-Income Ratio"].copy()

    cleaned["Debt-To-Income Ratio"] = parse_dti(
        cleaned["Debt-To-Income Ratio"]
    )

    add_log(
        cleaning_log,
        "Cleaning",
        "Debt-To-Income Ratio",
        "Percentage text format",
        "Removed % symbol and converted to numeric",
        int(
            before_dti.notna().sum() -
            cleaned["Debt-To-Income Ratio"].notna().sum()
        ),
        "No values were filled. Negative and high values are preserved."
    )

    # D. Convert numeric columns safely.
    numeric_columns = [
        "Amount Requested",
        "Risk_Score",
        "Policy Code"
    ]

    for col in numeric_columns:

        before = cleaned[col].copy()

        cleaned[col] = pd.to_numeric(
            cleaned[col],
            errors="coerce"
        )

        failures = int(
            before.notna().sum() -
            cleaned[col].notna().sum()
        )

        add_log(
            cleaning_log,
            "Cleaning",
            col,
            "Numeric data type",
            "Converted to numeric; invalid strings become null",
            failures,
            "No numeric imputation was performed."
        )

    # E. Standardize state abbreviations.
    before_state = cleaned["State"].copy()

    cleaned["State"] = (
        cleaned["State"]
        .astype("string")
        .str.upper()
    )

    state_changes = int(
        (
            before_state.fillna("<NA>").astype(str) !=
            cleaned["State"].fillna("<NA>").astype(str)
        ).sum()
    )

    add_log(
        cleaning_log,
        "Cleaning",
        "State",
        "Inconsistent letter case",
        "Converted state codes to uppercase",
        state_changes,
        "No state values were inferred or replaced."
    )

    # F. Keep ZIP codes as text to preserve leading zeros.
    cleaned["Zip Code"] = (
        cleaned["Zip Code"]
        .astype("string")
        .str.strip()
    )

    # G. Do NOT delete rows, columns, or fill missing values.
    add_log(
        cleaning_log,
        "Cleaning",
        "All Columns",
        "Missing values",
        "Kept all missing values as null",
        int(cleaned.isna().sum().sum()),
        "No automatic filling or row deletion."
    )

    add_log(
        cleaning_log,
        "Cleaning",
        "All Rows",
        "Duplicate rows",
        "Preserved duplicate rows",
        int(cleaned.duplicated().sum()),
        "No deletion without documented approval."
    )

    # --------------------------------------------------------
    # CELL 13 - FINAL ISSUE FLAGS
    # Flags are added for review, not for automatic deletion.
    # --------------------------------------------------------

    cleaned["Flag_Amount_Non_Positive"] = (
        cleaned["Amount Requested"] <= 0
    )

    cleaned["Flag_Risk_Score_Zero"] = (
        cleaned["Risk_Score"] == 0
    )

    cleaned["Flag_Risk_Score_Outside_300_850"] = (
        (cleaned["Risk_Score"] < 300) |
        (cleaned["Risk_Score"] > 850)
    )

    cleaned["Flag_DTI_Negative"] = (
        cleaned["Debt-To-Income Ratio"] < 0
    )

    cleaned["Flag_DTI_Above_100"] = (
        cleaned["Debt-To-Income Ratio"] > 100
    )

    cleaned["Flag_DTI_Extreme"] = (
        cleaned["Debt-To-Income Ratio"] > 1000
    )

    cleaned["Flag_Missing_Risk_Score"] = (
        cleaned["Risk_Score"].isna()
    )

    cleaned["Flag_Missing_Employment_Length"] = (
        cleaned["Employment Length"].isna()
    )

    cleaned["Flag_Missing_Loan_Title"] = (
        cleaned["Loan Title"].isna()
    )

    # --------------------------------------------------------
    # AUTOMATIC COLUMN REMOVAL BASED ON MISSINGNESS
    # Remove columns above min(average column missingness, 39%).
    # Threshold is calculated from original input columns only.
    # --------------------------------------------------------

    original_missing_pct = raw.isna().mean() * 100
    average_missing_pct = float(original_missing_pct.mean())
    missingness_cutoff = min(average_missing_pct, 39.0)
    columns_to_drop = original_missing_pct[
        original_missing_pct > missingness_cutoff
    ].index.tolist()

    # Remove review flags that belong to source columns being removed.
    flag_source_map = {
        "Amount Requested": ["Flag_Amount_Non_Positive"],
        "Risk_Score": [
            "Flag_Risk_Score_Zero",
            "Flag_Risk_Score_Outside_300_850",
            "Flag_Missing_Risk_Score"
        ],
        "Debt-To-Income Ratio": [
            "Flag_DTI_Negative", "Flag_DTI_Above_100", "Flag_DTI_Extreme"
        ],
        "Employment Length": ["Flag_Missing_Employment_Length"],
        "Loan Title": ["Flag_Missing_Loan_Title"]
    }
    related_flags = [
        flag for col in columns_to_drop
        for flag in flag_source_map.get(col, [])
    ]
    cleaned.drop(
        columns=columns_to_drop + related_flags,
        errors="ignore",
        inplace=True
    )

    missingness_report = pd.DataFrame({
        "Column": original_missing_pct.index,
        "Missing_Percentage": original_missing_pct.values,
        "Average_Missing_Percentage": average_missing_pct,
        "Applied_Cutoff_Percentage": missingness_cutoff,
        "Removed": [c in columns_to_drop for c in original_missing_pct.index]
    }).sort_values("Missing_Percentage", ascending=False)

    missingness_report.to_csv(
        os.path.join(output_dir, "Missingness_Column_Removal_Report.csv"),
        index=False
    )

    if columns_to_drop:
        add_log(
            cleaning_log,
            "Cleaning",
            ", ".join(columns_to_drop),
            "High column missingness",
            "Removed columns above the missingness cutoff",
            0,
            f"Cutoff = min(average column missingness {average_missing_pct:.2f}%, 39%) = {missingness_cutoff:.2f}%."
        )

    # --------------------------------------------------------
    # CELL 14 - FINAL QUALITY REPORTS
    # --------------------------------------------------------

    final_missing = pd.DataFrame({
        "Column": cleaned.columns,
        "Data_Type": cleaned.dtypes.astype(str).values,
        "Missing_Count": cleaned.isna().sum().values,
        "Missing_Percentage": (
            cleaned.isna().mean().values * 100
        ).round(2),
        "Unique_Values": cleaned.nunique().values
    }).sort_values(
        "Missing_Percentage",
        ascending=False
    )

    final_missing.to_csv(
        os.path.join(output_dir, "Final_Missing_Report.csv"),
        index=False
    )

    cleaning_log_df = pd.DataFrame(cleaning_log)

    cleaning_log_df.to_csv(
        os.path.join(output_dir, "Cleaning_Log.csv"),
        index=False
    )

    # Compare original and cleaned shape.
    summary = pd.DataFrame({
        "Metric": [
            "Original Rows",
            "Cleaned Rows",
            "Original Columns",
            "Cleaned Columns",
            "Original Duplicate Rows",
            "Cleaned Duplicate Rows",
            "Original Missing Cells",
            "Cleaned Missing Cells",
            "Raw File Modified"
        ],
        "Value": [
            raw.shape[0],
            cleaned.shape[0],
            raw.shape[1],
            cleaned.shape[1],
            int(raw.duplicated().sum()),
            int(cleaned.duplicated().sum()),
            int(raw.isna().sum().sum()),
            int(cleaned.isna().sum().sum()),
            "No"
        ]
    })

    summary.to_csv(
        os.path.join(output_dir, "Cleaning_Summary.csv"),
        index=False
    )

    # --------------------------------------------------------
    # CELL 15 - EXPORT RESULTS
    # --------------------------------------------------------

    cleaned_file = os.path.join(CLEANED_DIR, "rejected_cleaned.csv")
    cleaned.to_csv(cleaned_file, index=False)

    with pd.ExcelWriter(
        os.path.join(output_dir, "Rejected_Data_Quality_Report.xlsx"),
        engine="openpyxl"
    ) as writer:

        summary.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        null_report.to_excel(
            writer,
            sheet_name="Original Missing",
            index=False
        )

        final_missing.to_excel(
            writer,
            sheet_name="Final Missing",
            index=False
        )

        missingness_report.to_excel(
            writer,
            sheet_name="Column Removal",
            index=False
        )

        outlier_summary.to_excel(
            writer,
            sheet_name="Outlier Summary",
            index=False
        )

        category_report.to_excel(
            writer,
            sheet_name="Category Issues",
            index=False
        )

        redundant_report.to_excel(
            writer,
            sheet_name="Redundant Pairs",
            index=False
        )

        monthly.to_excel(
            writer,
            sheet_name="Monthly Timeline",
            index=False
        )

        yearly.to_excel(
            writer,
            sheet_name="Yearly Timeline",
            index=False
        )

        cleaning_log_df.to_excel(
            writer,
            sheet_name="Cleaning Log",
            index=False
        )

    guide_path = os.path.join(REPORT_ROOT, "Rejected_Cleaning_Guide.md")
    guide_text = f"""# Rejected Loans — Data Cleaning Guide

## Input and output
- Input: `data/Sampled/sample_rejected_2007_to_2018Q4.csv`
- Cleaned dataset: `data/Cleaned/rejected_cleaned.csv`
- Reports and charts: `docs/Data Cleaning/Rejected/50K_Sample/`

## What this notebook does
1. Profiles missing values and co-missingness.
2. Checks numeric outliers and domain-review conditions.
3. Identifies inconsistent category labels and potentially redundant columns.
4. Builds monthly and yearly issue timelines using Application Date.
5. Trims and normalizes whitespace in text fields while preserving capitalization.
6. Converts Application Date to datetime.
7. Converts Debt-To-Income Ratio to numeric after removing percent signs and commas.
8. Converts Amount Requested, Risk_Score, and Policy Code to numeric.
9. Converts State codes to uppercase and keeps Zip Code as text.
10. Adds review flags for selected suspicious or missing values.
11. Removes source columns whose missing-value percentage is strictly greater than the lower of the average column missingness and 39%.
12. Exports the cleaned CSV, quality workbook, cleaning log, summary, and column-removal report.

## Important interpretation
- The sampled source CSV is never overwritten.
- Missing values are not imputed.
- Duplicate rows and unusual numeric values are reported/flagged, not automatically deleted.
- The column-removal cutoff is calculated from the original input columns; columns exactly at the cutoff are retained.
- Review the column-removal report and cleaning log before using the output for analysis.
"""
    with open(guide_path, "w", encoding="utf-8") as guide_file:
        guide_file.write(guide_text)

    print("\nPIPELINE COMPLETED")
    print("Dataset:", dataset_name)
    print("Original shape:", raw.shape)
    print("Cleaned shape:", cleaned.shape)
    print("Cleaned CSV:", cleaned_file)
    print("Cleaning guide:", guide_path)
    print("Raw file modified: NO")
    print("Reports folder:", output_dir)

    return cleaned, cleaning_log_df


# ============================================================
# CELL 16 - RUN ON 50K / VALIDATION / SCALING / FULL DATASET
# ============================================================

results = {}

for dataset_name, input_path in INPUT_FILES.items():

    result = run_data_quality_pipeline(
        dataset_name,
        input_path
    )

    if result is not None:
        results[dataset_name] = result


print("\nALL AVAILABLE DATASETS PROCESSED.")
